# Minimal prepared XYZ waveform
Reuse the geometry and TDI operators for the fastest implemented repeat-evaluation path (the green bars). This small example generates one day at 5 s cadence with analytic LISA orbits, CPU float64, and generation-2 TDI. Run with a Python environment containing this package and pyTDI.


In [1]:
from pathlib import Path
from dataclasses import replace
from time import perf_counter
import sys
import numpy as np
import jax

# Allow running from either the repository root or notebooks/.
root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
if (root / "src").is_dir():
    sys.path.insert(0, str(root / "src"))
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")

from egb_jax_eccentric import (
    EccentricBinaryParams, lisa_orbit, precompute_jax_link_geometry,
    eccentric_links_jax, prepare_xyz_from_links,
)


In [2]:
# Once per time grid and orbit: prepare reusable geometry and TDI.
times = np.arange(0.0, 86400.0, 5.0)  # seconds
state = lisa_orbit(times)
geometry = precompute_jax_link_geometry(state)
tdi = prepare_xyz_from_links(
    state, generation=2, measurement_order=3, delay_order=3,
)

source = EccentricBinaryParams(
    mean_motion=np.pi * 1e-3,  # 1 mHz reference GW frequency
    eccentricity=0.3, m1_solar=0.6, m2_solar=0.4,
    distance_m=3.085677581491367e19,  # 1 kpc
    beta=0.2, lambda_=0.4, psi=0.3, inclination=0.8,
    phi0=0.2, fdot=0.0,
)


In [3]:
def waveform(source):
    links = eccentric_links_jax(
        source, geometry, batch_size=1, physics_mode="1pn_periastron",
    )
    return tdi(links)  # NumPy arrays: X, Y, Z; JAX work is synchronized

xyz = waveform(source)  # First call includes JIT warmup.

# Change source parameters while reusing the same prepared operators.
start = perf_counter()
xyz = waveform(replace(source, eccentricity=0.4))
elapsed = perf_counter() - start
assert all(np.isfinite(values).all() for values in xyz.values())
print({channel: values.shape for channel, values in xyz.items()})
print(f"Warm waveform + TDI: {elapsed:.4f} s (preparation excluded)")


{'X': (17280,), 'Y': (17280,), 'Z': (17280,)}
Warm waveform + TDI: 0.0117 s (preparation excluded)


`xyz` contains the dimensionless fractional-frequency Michelson channels. The timed call includes fresh waveform/link generation and TDI application. Rebuild the caches if the grid, orbit, or interpolation settings change; use shared blocks for long observations to limit cache memory.

This uses fixed eccentricity with 1PN periastron advance, without Peters–Mathews evolution. It is a calling example, not a cadence-convergence or Sangria-convention validation. Finite-grid TDI boundaries require padding for production segments. pyTDI may emit its existing complex-to-real warning for real GW links stored in complex arrays.
